# 01. Ekstraksi Data Mentah OJK & *Feature Engineering*
## (*Zero-Jump Data Provenance*: `data/raw/` $\rightarrow$ `data/processed/`)
---
**Tujuan Notebook:** Menunjukkan langkah demi langkah bagaimana berkas statistik publikasi resmi **OJK Desember 2025** (`pinjaman_fintech_ojk_des2025.xlsx`) dibaca dari bentuk mentahnya (*raw XML/Excel*), dibersihkan, ditambahkan fitur turunan (*feature engineering*), direkonsiliasi secara matematis, hingga menghasilkan 3 berkas CSV siap pakai di `data/processed/` untuk dijalankan oleh **Apache Airflow DAG 1–3**.

**Alur Tanpa Lompat Langkah:**
1. Inspeksi struktur 24 *sheet* pada Excel mentah OJK.
2. Parsing & pembersihan **Sheet 4** (rasio kinerja keuangan bulanan).
3. *Feature engineering* indikator kepatuhan (`risk_level`, `alert_trigger`) $\rightarrow$ `ojk_monthly_summary.csv`.
4. Ekstraksi **Sheet 18 Kolom O** (38 provinsi), *feature engineering* konsentrasi pasar (`share_nasional_persen`, `risk_tier`), dan rekonsiliasi total nasional $\rightarrow$ `ojk_regional_exposure.csv`.
5. Kalibrasi parameter empiris OJK + aturan **POJK 10/POJK.05/2022** untuk membentuk data batch operasional `fintech_loan_daily_batch.csv` (1.000 akun peminjam) beserta fitur konsistensi waktu (`disbursement_date`) dan flag `is_twp90_breach`.

In [ ]:
# Setup library standar (zero heavy Excel dependency) & resolusi path portabel
import os
import csv
import zipfile
import random
import datetime
import xml.etree.ElementTree as ET
import pandas as pd

CWD = os.getcwd()
PARENT_DIR = os.path.dirname(CWD)

# Deteksi root proyek apakah notebook dijalankan dari root repo atau folder notebooks/
ROOT_DIR = next(
    (d for d in (CWD, PARENT_DIR) if os.path.isdir(os.path.join(d, 'data')) or os.path.isdir(os.path.join(d, 'Dataset_OJK'))),
    PARENT_DIR,
)
RAW_PATH = next(
    (p for p in (
        os.path.join(ROOT_DIR, 'data', 'raw', 'pinjaman_fintech_ojk_des2025.xlsx'),
        os.path.join(ROOT_DIR, 'pinjaman_fintech_ojk_des2025.xlsx'),
        os.path.join(ROOT_DIR, 'Dataset_OJK', 'pinjaman_fintech_ojk_des2025.xlsx'),
        os.path.join(ROOT_DIR, '02_Dataset_Fintech_OJK', 'pinjaman_fintech_ojk_des2025.xlsx'),
    ) if os.path.exists(p)),
    os.path.join(ROOT_DIR, 'data', 'raw', 'pinjaman_fintech_ojk_des2025.xlsx'),
)
PROCESSED_DIR = next(
    (p for p in (
        os.path.join(ROOT_DIR, 'data', 'processed'),
        os.path.join(ROOT_DIR, 'Dataset_OJK'),
        os.path.join(ROOT_DIR, '02_Dataset_Fintech_OJK'),
    ) if os.path.isdir(p)),
    os.path.join(ROOT_DIR, 'data', 'processed'),
)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f'[OK] RAW_PATH     : {RAW_PATH} (exists={os.path.exists(RAW_PATH)})')
print(f'[OK] PROCESSED_DIR: {PROCESSED_DIR}')

## Tahap 1: Inspeksi Struktur Mentah Berkas Excel OJK
Berkas `.xlsx` sebenarnya merupakan arsip ZIP berisi dokumen XML OOXML. Dengan membaca `xl/workbook.xml`, `xl/_rels/workbook.xml.rels`, dan `xl/sharedStrings.xml`, kita dapat memeriksa ke-24 *sheet* beserta sel mentahnya secara transparan tanpa ketergantungan *library* eksternal.

In [ ]:
def parse_shared_strings(z):
    """Membaca xl/sharedStrings.xml per elemen <si> agar indeks string presisi."""
    strings = []
    if 'xl/sharedStrings.xml' not in z.namelist():
        return strings
    tree = ET.fromstring(z.read('xl/sharedStrings.xml'))
    for si in tree.iter():
        if si.tag.endswith('si'):
            parts = [t.text for t in si.iter() if t.tag.endswith('t') and t.text]
            strings.append(''.join(parts))
    return strings

def parse_sheet_rows(ws_xml, shared_strings):
    """Mengonversi worksheet XML menjadi dict {row_idx: {col_letter: raw_value}}."""
    rows = {}
    ns_v = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v'
    for c in ws_xml.iter():
        if c.tag.endswith('c'):
            r_ref = c.attrib.get('r', '')
            if not r_ref:
                continue
            r_idx = int(''.join(filter(str.isdigit, r_ref)))
            c_ref = ''.join(filter(str.isalpha, r_ref))
            t = c.attrib.get('t')
            v = c.find(ns_v)
            val = ''
            if v is not None and v.text is not None:
                val = shared_strings[int(v.text)] if t == 's' else v.text
            rows.setdefault(r_idx, {})[c_ref] = val
    return rows

with zipfile.ZipFile(RAW_PATH, 'r') as z:
    shared_strings = parse_shared_strings(z)
    wb = ET.fromstring(z.read('xl/workbook.xml'))
    rels = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
    rel_map = {r.attrib['Id']: r.attrib['Target'] for r in rels.iter() if r.tag.endswith('Relationship')}
    rel_attr = '{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id'

    sheet_targets = {}
    for s in wb.iter():
        if s.tag.endswith('sheet'):
            name = s.attrib['name'].strip()
            target = rel_map[s.attrib[rel_attr]].lstrip('/')
            if not target.startswith('xl/'):
                target = 'xl/' + target
            sheet_targets[name] = target

    rows4 = parse_sheet_rows(ET.fromstring(z.read(sheet_targets['4'])), shared_strings)
    rows18 = parse_sheet_rows(ET.fromstring(z.read(sheet_targets['18'])), shared_strings)

print(f'Total Sheet ditemukan: {len(sheet_targets)} -> {list(sheet_targets.keys())}')
print('\nContoh baris mentah Sheet 4 (Kolom A = Label, Kolom B..N = Serial Date & Rasio Desimal):')
for r in [2, 3, 4, 5, 6]:
    print(f'  Row {r}:', {k: rows4[r][k] for k in sorted(rows4[r], key=lambda c: (len(c), c)) if k in {'A','B','M','N'}})

print('\nContoh baris mentah Sheet 18 Kolom O (Des-25):')
for r in [4, 5, 6, 7, 11, 40, 41, 42, 43, 44]:
    label = rows18.get(r, {}).get('B') or rows18.get(r, {}).get('A')
    print(f'  Row {r:2d}: Label={label!r:30s} | O={rows18.get(r, {}).get("O")!r}')

## Tahap 2: Parsing & Pembersihan Sheet 4 (Data Seri Waktu Bulanan)
Pada data mentah di atas:
* Baris 2 menyimpan **Excel Serial Date** (`45627` = `2024-12` s.d. `46021` = `2025-12`). Konversi dilakukan dari epoch Excel `1899-12-30`.
* Baris 3–6 menyimpan rasio desimal (`0.9568`), sehingga dikalikan `100` menjadi skala persentase (`95.68`).
* Kolom Excel diurutkan berdasarkan panjang huruf lalu alfabet `(len(col), col)` agar urutan waktu kronologis aman.

In [ ]:
date_cols = [
    c for c in sorted(rows4[2].keys(), key=lambda col: (len(col), col))
    if rows4[2][c].isdigit()
]

monthly_base = []
for c in date_cols:
    serial = int(rows4[2][c])
    dt = datetime.date(1899, 12, 30) + datetime.timedelta(days=serial)
    tkb90 = float(rows4[3].get(c, 0)) * 100
    twp90 = float(rows4[4].get(c, 0)) * 100
    roa = float(rows4[5].get(c, 0)) * 100 if c in rows4.get(5, {}) and rows4[5][c] else 0.0
    roe = float(rows4[6].get(c, 0)) * 100 if c in rows4.get(6, {}) and rows4[6][c] else 0.0

    monthly_base.append({
        'periode': dt.strftime('%Y-%m'),
        'tkb90_persen': round(tkb90, 2),
        'twp90_persen': round(twp90, 2),
        'roa_persen': round(roa, 2),
        'roe_persen': round(roe, 2),
    })

df_monthly_base = pd.DataFrame(monthly_base)
print('Tabel bersih Sheet 4 sebelum Feature Engineering:')
df_monthly_base

## Tahap 3: *Feature Engineering* Data Bulanan $\rightarrow$ `ojk_monthly_summary.csv`
Dari metrik dasar di atas, diturunkan **2 fitur orkestrasi** untuk mendukung *Decision Gate* pada Airflow DAG 2:
1. **`risk_level`:** Klasifikasi kehati-hatian OJK (`WASPADA` jika `TWP90 > 4.0%`, `PERHATIAN` jika `> 3.0%`, selain itu `NORMAL`).
2. **`alert_trigger`:** Flag biner (`1` jika melampaui toleransi `4.0%`, else `0`) yang menandakan kondisi pemicuan darurat ke DAG 3.

In [ ]:
monthly_summary = []
for row in monthly_base:
    twp90 = row['twp90_persen']
    risk_level = 'WASPADA' if twp90 > 4.0 else ('PERHATIAN' if twp90 > 3.0 else 'NORMAL')
    alert_trigger = 1 if twp90 > 4.0 else 0
    monthly_summary.append({**row, 'risk_level': risk_level, 'alert_trigger': alert_trigger})

csv1_path = os.path.join(PROCESSED_DIR, 'ojk_monthly_summary.csv')
with open(csv1_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['periode', 'tkb90_persen', 'twp90_persen', 'roa_persen', 'roe_persen', 'risk_level', 'alert_trigger'])
    writer.writeheader()
    writer.writerows(monthly_summary)

print(f'[OK] Tersimpan {len(monthly_summary)} baris -> {csv1_path}')
pd.DataFrame(monthly_summary)

## Tahap 4: Ekstraksi, *Feature Engineering*, & Rekonsiliasi Wilayah (Sheet 18 Kolom O)
Dari **Sheet 18 Kolom O** (posisi tutup tahun **Desember 2025**):
* Baris `5–10` adalah **6 Provinsi di Pulau Jawa** (Subtotal Baris 4 = Rp 1.068.292,95 Miliar).
* Baris `12–43` adalah **32 Provinsi di Luar Jawa** termasuk 4 Provinsi DOB baru Papua pada Baris `40–43` (Subtotal Baris 11 = Rp 320.544,42 Miliar).
* Baris `44` adalah Total Akumulasi Nasional (**Rp 1.388.837,37 Miliar**).

**Fitur turunan (*feature engineering*) yang dibentuk:**
* **`share_nasional_persen`**: $(\text{penyaluran\_miliar} / \text{Total Nasional Baris 44}) \times 100\%$
* **`risk_tier`**: `TINGGI` (>10%), `SEDANG` (2%–10%), `RENDAH` (<2%) untuk memetakan konsentrasi risiko portofolio geografis.

In [ ]:
total_nasional = float(rows18.get(44, {}).get('O', 1388837.37))

province_map = [
    (5, 'Jawa', '1. Banten'),
    (6, 'Jawa', '2. DKI Jakarta'),
    (7, 'Jawa', '3. Jawa Barat'),
    (8, 'Jawa', '4. Jawa Tengah'),
    (9, 'Jawa', '5. DI Yogyakarta'),
    (10, 'Jawa', '6. Jawa Timur'),
    (12, 'Luar Jawa', '1. Nangroe Aceh Darussalam'),
    (13, 'Luar Jawa', '2. Sumatera Utara'),
    (14, 'Luar Jawa', '3. Sumatera Barat'),
    (15, 'Luar Jawa', '4. Riau'),
    (16, 'Luar Jawa', '5. Kepulauan Riau'),
    (17, 'Luar Jawa', '6. Kepualauan Bangka Belitung'),
    (18, 'Luar Jawa', '7. Jambi'),
    (19, 'Luar Jawa', '8. Sumatera Selatan'),
    (20, 'Luar Jawa', '9. Bengkulu'),
    (21, 'Luar Jawa', '10. Lampung'),
    (22, 'Luar Jawa', '11. Kalimantan Barat'),
    (23, 'Luar Jawa', '12. Kalimantan Tengah'),
    (24, 'Luar Jawa', '13. Kalimantan Utara'),
    (25, 'Luar Jawa', '14. Kalimantan Timur'),
    (26, 'Luar Jawa', '15. Kalimantan Selatan'),
    (27, 'Luar Jawa', '16. Sulawesi Utara'),
    (28, 'Luar Jawa', '17. Gorontalo'),
    (29, 'Luar Jawa', '18. Sulawesi Tengah'),
    (30, 'Luar Jawa', '19. Sulawesi Barat'),
    (31, 'Luar Jawa', '20. Sulawesi Selatan'),
    (32, 'Luar Jawa', '21. Sulawesi Tenggara'),
    (33, 'Luar Jawa', '22. Bali'),
    (34, 'Luar Jawa', '23. Nusa Tenggara Barat'),
    (35, 'Luar Jawa', '24. Nusa Tenggara Timur'),
    (36, 'Luar Jawa', '25. Maluku Utara'),
    (37, 'Luar Jawa', '26. Maluku'),
    (38, 'Luar Jawa', '27. Papua Barat'),
    (39, 'Luar Jawa', '28. Papua'),
    (40, 'Luar Jawa', '29. Papua Tengah'),
    (41, 'Luar Jawa', '30. Papua Pegunungan'),
    (42, 'Luar Jawa', '31. Papua Selatan'),
    (43, 'Luar Jawa', '32. Papua Barat Daya'),
]

regional_rows = []
clean_province_names = []
province_weights = []

for r_idx, region, prov_name in province_map:
    val_o = rows18.get(r_idx, {}).get('O', '')
    if not val_o:
        continue
    penyaluran = float(val_o)
    share = round((penyaluran / total_nasional) * 100, 2)
    tier = 'TINGGI' if share > 10.0 else ('SEDANG' if share > 2.0 else 'RENDAH')

    regional_rows.append({
        'periode': '2025-12',
        'wilayah': region,
        'nama_provinsi': prov_name,
        'penyaluran_miliar': round(penyaluran, 2),
        'share_nasional_persen': share,
        'risk_tier': tier,
    })
    clean_province_names.append(prov_name.split('. ', 1)[-1])
    province_weights.append(share)

csv2_path = os.path.join(PROCESSED_DIR, 'ojk_regional_exposure.csv')
with open(csv2_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['periode', 'wilayah', 'nama_provinsi', 'penyaluran_miliar', 'share_nasional_persen', 'risk_tier'])
    writer.writeheader()
    writer.writerows(regional_rows)

df_regional = pd.DataFrame(regional_rows)
sum_38 = round(df_regional['penyaluran_miliar'].sum(), 2)
total_nasional_2dp = round(total_nasional, 2)
assert sum_38 == total_nasional_2dp, f'Mismatch: {sum_38} != {total_nasional_2dp}'
print(f'[OK] Rekonsiliasi 38 provinsi cocok 100%: Rp {sum_38:,.2f} Miliar == Total Baris 44 OJK (Rp {total_nasional_2dp:,.2f} Miliar)')
df_regional.sort_values('penyaluran_miliar', ascending=False).head(10)

## Tahap 5: Kalibrasi Parameter & Generasi Data Batch Operasional (`fintech_loan_daily_batch.csv`)
Karena publikasi OJK berbentuk tabel agregat (demi kerahasiaan nasabah/*bank secrecy*), sedangkan **Quality Gate Airflow DAG 1** menguji validasi skema tingkat rekening (*loan-level*), tahap ini menurunkan 1.000 baris transaksi operasional yang dikalibrasi langsung ke parameter resmi OJK:
* **Distribusi Provinsi (`province`):** *Weighted Random Sampling* berbasis bobot `share_nasional_persen` dari Tahap 4 (Sheet 18).
* **Demografi Peminjam (`age_group`):** Diselaraskan dengan dominasi usia produktif 19–34 tahun (~60%) sesuai Sheet 11 OJK.
* **Kolektibilitas (`loan_status`, `days_overdue`, `is_twp90_breach`):** Mematuhi 5 kelas kolektibilitas **POJK No. 10/POJK.05/2022**.
* **Konsistensi Kronologis (`disbursement_date`):** Dihitung mundur dari tanggal snapshot `2025-12-30` dikurangi `(tenor_months * 30 + days_overdue)` untuk pinjaman menunggak agar tidak terjadi anomali tanggal cair lebih muda daripada lama tunggakan.

In [ ]:
snapshot_date = datetime.date(2025, 12, 30)
random.seed(42)
transactions = []

age_groups = ['<19', '19-34', '35-54', '>54']
age_weights = [0.05, 0.60, 0.30, 0.05]
genders = ['L', 'P']
statuses = ['LANCAR', 'DALAM_PERHATIAN', 'KURANG_LANCAR', 'DIRAGUKAN', 'MACET']
status_weights = [0.85, 0.08, 0.03, 0.02, 0.02]

for i in range(1, 1001):
    status = random.choices(statuses, weights=status_weights)[0]
    tenor = random.choice([1, 3, 6, 12])

    if status == 'LANCAR':
        overdue = 0
        disburse_days_ago = random.randint(10, 90)
    elif status == 'DALAM_PERHATIAN':
        overdue = random.randint(1, 30)
        disburse_days_ago = (tenor * 30) + overdue
    elif status == 'KURANG_LANCAR':
        overdue = random.randint(31, 60)
        disburse_days_ago = (tenor * 30) + overdue
    elif status == 'DIRAGUKAN':
        overdue = random.randint(61, 90)
        disburse_days_ago = (tenor * 30) + overdue
    else:  # MACET
        overdue = random.randint(91, 180)
        disburse_days_ago = (tenor * 30) + overdue

    disburse_dt = snapshot_date - datetime.timedelta(days=disburse_days_ago)
    prov = random.choices(clean_province_names, weights=province_weights, k=1)[0]

    transactions.append({
        'loan_id': f'LN202512{i:04d}',
        'borrower_id': f'BRW-{random.randint(10000, 99999)}',
        'disbursement_date': disburse_dt.strftime('%Y-%m-%d'),
        'age_group': random.choices(age_groups, weights=age_weights)[0],
        'gender': random.choice(genders),
        'province': prov,
        'loan_amount': random.choice([1500000, 2500000, 3000000, 5000000, 8000000, 10000000]),
        'tenor_months': tenor,
        'interest_rate_pct': round(random.uniform(7.5, 12.0), 1),
        'loan_status': status,
        'days_overdue': overdue,
        'is_twp90_breach': 1 if overdue > 90 else 0,
    })

csv3_path = os.path.join(PROCESSED_DIR, 'fintech_loan_daily_batch.csv')
fieldnames_batch = [
    'loan_id', 'borrower_id', 'disbursement_date', 'age_group', 'gender',
    'province', 'loan_amount', 'tenor_months', 'interest_rate_pct',
    'loan_status', 'days_overdue', 'is_twp90_breach',
]
with open(csv3_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames_batch)
    writer.writeheader()
    writer.writerows(transactions)

df_batch = pd.DataFrame(transactions)
print(f'[OK] Tersimpan {len(df_batch)} transaksi -> {csv3_path}')
print('\nDistribusi Status Kolektibilitas:')
print(df_batch['loan_status'].value_counts())
df_batch.head()

## Tahap 6: Jembatan Menuju Pipeline Apache Airflow & Visualisasi
Ketiga berkas hasil ekstraksi dan *feature engineering* di `data/processed/` kini siap diproses oleh:
1. **`dags/dag_1_ojk_ingest_validate.py`**: Memvalidasi integritas skema dan konsistensi kolektibilitas sebelum memicu DAG 2 via `TriggerDagRunOperator`.
2. **`dags/dag_2_risk_portfolio_analytics.py`**: Mengagregasi Data Mart ke `data/output/ojk_portfolio_risk_mart.csv` dan mengevaluasi ambang batas TWP90 via `BranchPythonOperator`.
3. **`dags/dag_3_compliance_alert_automation.py`**: Mencatat laporan insiden ke `data/output/compliance_alert_log.txt` saat TWP90 > 4,00%.
4. **`notebooks/02_analytics_and_pipeline_dashboard.ipynb`**: Melakukan analisis visual dan verifikasi Data Mart hasil eksekusi Airflow.